# ECG-image inference and Grad-CAM in Google Colab

**Version v0.4.0.** The source-loading cell selects `v0.4.0` and uses the unchanged **v0.2.0 weights**. The candidate workflow completed Colab CPU/A100 checks on 2026-10-29. The final run starting from the public v0.4.0 tag completed on 2026-10-29 using the generated image, A100, paper TTA and Grad-CAM; see the validation section for the checked scope.

Choose a synthetic example, upload one image from your computer, or read an image from Google Drive. Select how an image is prepared for the model, then run an individual model or the raw-logit ensemble. The default requires **480 x 480** input and uses the paper TTA settings. Grad-CAM is enabled by default for ViT and ConvNeXtV2.

The device selector prefers a GPU. If GPU execution is unavailable, a warning is displayed and inference uses **CPU**. CPU can be slower; select **No TTA** or fewer **Custom** views if desired. TTA settings are never reduced automatically. Setup installs the matching CUDA or CPU PyTorch build in a separate inference environment.

The generated trace is a software example without clinical meaning. Review the prepared input before running inference, especially after cropping or padding. Review the weight terms before downloading; the complete bundle includes the ViT noncommercial condition. Save the matching predictions, run record and optional visualization files using the final download cell.

## 1. Load the source release


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import os
import re
import subprocess
import sys

# A failed source-selection attempt must invalidate earlier success state.
ENVIRONMENT_READY = False
WEIGHTS_READY = False
DEVICE_READY = False
INPUT_READY = False
OPTIONS_READY = False
IMAGE_PATH = None
PREPARED_IMAGE_PATH = None
input_record = None
RESULT_INPUT_TOKEN = None
RESULT_OPTIONS_SIGNATURE = None
result = None

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Colab runtime with Python 3.11 or later.")

REPO_URL = "https://github.com/ekagawa007/takotsubo_ami_ai_ecg.git"
GIT_REF = "v0.4.0"  # Source release used by this notebook.
REPO_DIR = Path("/content/takotsubo_ami_ai_ecg")
PACKAGE_SOURCE_COMMIT = globals().get("_VERIFIED_PACKAGE_SOURCE_COMMIT")

if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", GIT_REF):
    raise ValueError("Set GIT_REF to a branch name, tag name, or commit SHA.")
if REPO_DIR.is_symlink():
    raise RuntimeError("REPO_DIR must not be a symbolic link.")

def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / ".git").is_dir() or (REPO_DIR / ".git").is_symlink():
        raise RuntimeError("REPO_DIR is not a regular Git working directory. Choose another location.")
    if git("remote", "get-url", "origin").rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("The repository in REPO_DIR does not match REPO_URL.")
    if git("status", "--porcelain"):
        raise RuntimeError("The working directory has changes. Save them before rerunning this cell.")

try:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError("Could not load the requested source revision. Check the log and confirm GIT_REF is published.") from error
CODE_COMMIT = git("rev-parse", "HEAD")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise RuntimeError("Git did not return a complete commit identifier.")

required_names = [
    "requirements-inference.txt", "tools/download_weights.py",
    "src/takotsubo_ami_ai_ecg/checkpoint_compat.py",
    "src/takotsubo_ami_ai_ecg/gradcam.py",
    "src/takotsubo_ami_ai_ecg/security.py",
    "src/takotsubo_ami_ai_ecg/image_input.py",
    "src/takotsubo_ami_ai_ecg/safe_output.py",
    "src/takotsubo_ami_ai_ecg/progress.py",
    "src/takotsubo_ami_ai_ecg/assets/weights_v0.2.0.json",
    *[f"examples/predict_{name}.py" for name in ("ensemble", "vit", "convnextv2", "mlpmixer")],
]
required_names = sorted(set(required_names) | {
    p.relative_to(REPO_DIR).as_posix() for p in (REPO_DIR / "src/takotsubo_ami_ai_ecg").glob("*.py")
})
SOURCE_SHA256 = {}
for name in required_names:
    path = REPO_DIR / name
    if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
        raise RuntimeError("The selected source lacks a required regular inference file: " + name)
    SOURCE_SHA256[name] = hashlib.sha256(path.read_bytes()).hexdigest()

def verify_source():
    if REPO_DIR.is_symlink() or not REPO_DIR.is_dir() or git("rev-parse", "HEAD") != CODE_COMMIT:
        raise RuntimeError("The source checkout changed. Start a fresh runtime.")
    if git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("Tracked source files changed. Save your edits and start a fresh runtime.")
    for name, expected in SOURCE_SHA256.items():
        path = REPO_DIR / name
        if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
            raise RuntimeError("A source file is missing or is a symbolic link.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise RuntimeError("An inference source file changed. Start a fresh runtime.")

for module_name, module in tuple(sys.modules.items()):
    if module_name == "takotsubo_ami_ai_ecg" or module_name.startswith("takotsubo_ami_ai_ecg."):
        module_file = getattr(module, "__file__", None)
        if (PACKAGE_SOURCE_COMMIT != CODE_COMMIT
                or module_file and not Path(module_file).resolve().is_relative_to(REPO_DIR.resolve())):
            raise RuntimeError("A different package revision is loaded. Start a fresh Colab runtime.")
verify_source()
# Keep this binding unchanged if a later source-selection attempt fails.
_VERIFIED_PACKAGE_SOURCE_COMMIT = CODE_COMMIT
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:", CODE_COMMIT)

## 2. Select the device and create the inference environment

Keep **auto** to prefer an available GPU, or select **cpu** explicitly. If a requested GPU is unavailable, the notebook reports the fallback and uses CPU. Choose a GPU runtime in Colab when available. CPU requires no GPU allocation.

Setup creates a separate environment with matching PyTorch 2.13.0 / torchvision 0.28.0 CUDA 13.0 or CPU builds, transformers 5.10.4, Pillow 13.0.0 and filelock 4.2.1. The notebook kernel's ML packages are retained. Setup can take several minutes; a session restart is normally unnecessary.

After restarting, run from the first code cell. A completed matching environment is checked and reused. If installation fails partway, preserve the error log and start a fresh runtime. Changing the device selection requires rerunning setup and subsequent cells.

In [ ]:
# @title Select the device and prepare the inference environment
ENVIRONMENT_READY = False
DEVICE_READY = False
INPUT_READY = False
OPTIONS_READY = False
result = None
REQUESTED_DEVICE = "auto" # @param ["auto", "cuda", "cpu"]

from takotsubo_ami_ai_ecg.colab_support import choose_device

# Probe in a subprocess, without importing ML libraries into the notebook kernel.
try:
    probe = subprocess.run(
        [sys.executable, "-c", "import json, torch; print(json.dumps(torch.cuda.is_available()))"],
        capture_output=True, text=True, timeout=30,
    )
    detected_cuda = probe.returncode == 0 and json.loads(probe.stdout) is True
except (OSError, ValueError, subprocess.TimeoutExpired):
    detected_cuda = False
BUILD_DEVICE, device_notice = choose_device(REQUESTED_DEVICE, detected_cuda)
if device_notice:
    print("WARNING:", device_notice)
DEVICE_REQUEST_AT_SETUP = REQUESTED_DEVICE
WHEEL_INDEX = "https://download.pytorch.org/whl/" + ("cu130" if BUILD_DEVICE == "cuda" else "cpu")
VENV_DIR = Path(f"/content/ecg_inference_venv_{BUILD_DEVICE}")
INFERENCE_PYTHON = VENV_DIR / "bin" / "python"
READY_MARKER = VENV_DIR / "inference_ready.json"
build_suffix = "+cu130" if BUILD_DEVICE == "cuda" else "+cpu"
EXPECTED_VERSIONS = {
    "torch": "2.13.0" + build_suffix, "torchvision": "0.28.0" + build_suffix,
    "transformers": "5.10.4", "timm": "1.0.22", "numpy": "2.2.6",
    "Pillow": "13.0.0", "safetensors": "0.7.0", "setuptools": "84.2.1",
    "pip": "26.2.2", "filelock": "4.2.1",
}
EXPECTED_SETUP = {
    "code_commit": CODE_COMMIT, "requirements_sha256": SOURCE_SHA256["requirements-inference.txt"],
    "versions": EXPECTED_VERSIONS, "wheel_index": WHEEL_INDEX,
}
if VENV_DIR.is_symlink():
    raise RuntimeError("The inference environment path must not be a symbolic link.")
if VENV_DIR.exists():
    if not READY_MARKER.is_file() or READY_MARKER.is_symlink():
        raise RuntimeError("The environment is incomplete. Save the error log and use a fresh Colab runtime.")
    if json.loads(READY_MARKER.read_text(encoding="utf-8")) != EXPECTED_SETUP:
        raise RuntimeError("This environment belongs to a different source revision. Use a fresh runtime.")
else:
    pip_version = tuple(int(v) for v in metadata.version("pip").split(".")[:2])
    if pip_version < (22, 3):
        raise RuntimeError("This setup needs pip 22.3 or later. Select a current Colab runtime.")
    VENV_DIR.mkdir(exist_ok=False)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV_DIR)], check=True)
    managed_pip = [sys.executable, "-m", "pip", "--python", str(INFERENCE_PYTHON)]
    subprocess.run(managed_pip + ["install", "pip==26.2.2", "setuptools==84.2.1", "--index-url", "https://pypi.org/simple"], check=True)
    subprocess.run(managed_pip + ["install", "torch==2.13.0", "torchvision==0.28.0", "--index-url", WHEEL_INDEX], check=True)
    subprocess.run(managed_pip + ["install", "-r", str(REPO_DIR / "requirements-inference.txt"), "--index-url", "https://pypi.org/simple"], check=True)
    subprocess.run([str(INFERENCE_PYTHON), "-m", "pip", "check"], check=True)
    installed = json.loads(subprocess.check_output(
        [str(INFERENCE_PYTHON), "-c", "import importlib.metadata as m,json; print(json.dumps({k:m.version(k) for k in " + repr(tuple(EXPECTED_VERSIONS)) + "}))"],
        text=True,
    ))
    if installed != EXPECTED_VERSIONS:
        raise RuntimeError("Inference package versions differ from the selected versions.")
    from takotsubo_ami_ai_ecg.safe_output import write_files
    write_files([(READY_MARKER, (json.dumps(EXPECTED_SETUP, indent=2) + "\n").encode("utf-8"))])

subprocess.run([str(INFERENCE_PYTHON), "-m", "pip", "check"], check=True)
ENVIRONMENT_READY = True

def check_inference_environment():
    if not globals().get("ENVIRONMENT_READY", False):
        raise RuntimeError("Run the setup cells in order after opening or restarting the session.")
    if REQUESTED_DEVICE != DEVICE_REQUEST_AT_SETUP:
        raise RuntimeError("The device selection changed. Rerun environment setup and the following cells.")
    if not INFERENCE_PYTHON.is_file() or READY_MARKER.is_symlink():
        raise RuntimeError("The inference environment is unavailable.")
    if json.loads(READY_MARKER.read_text(encoding="utf-8")) != EXPECTED_SETUP:
        raise RuntimeError("The inference environment marker changed.")
    verify_source()

check_inference_environment()
print("Inference environment ready. Notebook-kernel ML packages were not replaced.")
print("Installed build:", BUILD_DEVICE)

In [ ]:
DEVICE_READY = False
result = None
check_inference_environment()
runtime_versions = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", 'import importlib.metadata as metadata\nimport json\nimport sys\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom transformers import ViTConfig, ViTForImageClassification\nversions = {"python": sys.version.split()[0]}\nfor package in ("torch", "torchvision", "timm", "transformers", "numpy", "Pillow", "safetensors", "setuptools", "pip", "filelock"):\n    versions[package] = metadata.version(package)\nversions["installed_packages"] = sorted([{ "name": d.metadata["Name"], "version": d.version } for d in metadata.distributions() if d.metadata.get("Name")], key=lambda item: item["name"].lower())\nversions["cuda_available"] = torch.cuda.is_available()\nversions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\nprint(json.dumps(versions))'], text=True,
))
if any(runtime_versions[name] != version for name, version in EXPECTED_VERSIONS.items()):
    raise RuntimeError("The inference interpreter has unexpected package versions.")
DEVICE, runtime_notice = choose_device(REQUESTED_DEVICE, runtime_versions["cuda_available"])
if runtime_notice:
    print("WARNING:", runtime_notice)
DEVICE_READY = True
RESOLVED_DEVICE = DEVICE

def check_execution_device():
    check_inference_environment()
    if not globals().get("DEVICE_READY", False) or DEVICE != RESOLVED_DEVICE:
        raise RuntimeError("Run the runtime check after selecting the execution device.")

print("Actual inference device:", DEVICE)
print(json.dumps({k: v for k, v in runtime_versions.items() if k != "installed_packages"}, indent=2))
print("Recorded installed distributions:", len(runtime_versions["installed_packages"]))

## 3. Prepare the weights

The default downloads the public v0.2.0 weights and verifies the archive and every bundled file's SHA-256. To use existing inference weights, set `WEIGHT_SOURCE = "local"` and edit `WEIGHTS_DIR`.

Enable the optional Drive connection below only for existing weights stored on Drive. Choosing **Google Drive** for the input image connects Drive in the input cell instead. The synthetic example and computer upload do not require Drive.

In [ ]:
# @title Optional Drive connection for existing weights
USE_GOOGLE_DRIVE = False # @param {type:"boolean"}
# Input images selected from Drive are connected automatically in the input cell.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
WEIGHTS_READY = False
result = None
check_inference_environment()

WEIGHT_SOURCE = "release"  # "release" for verified public weights; "local" for existing files.
WEIGHTS_DIR = Path("/content/ecg_inference_weights")
# Examples for WEIGHT_SOURCE = "local":
# WEIGHTS_DIR = Path("/content/weights")
# WEIGHTS_DIR = Path("/content/drive/MyDrive/ecg_inference/weights")

if WEIGHT_SOURCE not in {"release", "local"}:
    raise ValueError('Set WEIGHT_SOURCE to "release" or "local".')
if WEIGHT_SOURCE == "release":
    download_command = [
        str(INFERENCE_PYTHON), "-u", str(REPO_DIR / "tools" / "download_weights.py"),
        "--output-dir", str(WEIGHTS_DIR),
    ]
    download_process = None
    try:
        download_process = subprocess.Popen(
            download_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in download_process.stdout:
            print(line, end="", flush=True)
        download_returncode = download_process.wait()
        if download_returncode != 0:
            raise RuntimeError(
                f"Weight download or verification failed with exit code {download_returncode}. "
                "Review the error above. Check the v0.2.0 Release asset or select existing local weights."
            )
    except BaseException:
        if download_process is not None and download_process.poll() is None:
            download_process.terminate()
            try:
                download_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                download_process.kill()
                download_process.wait(timeout=5)
        raise
    finally:
        if download_process is not None and download_process.stdout is not None:
            download_process.stdout.close()
else:
    if not WEIGHTS_DIR.is_dir():
        raise FileNotFoundError("Set WEIGHTS_DIR to the existing inference-weight directory.")
    print("Using local inference weights; no download requested.")

WEIGHTS_READY = True
print("Weights directory:", WEIGHTS_DIR)

## 4. Choose and prepare the input image

Choose **Synthetic example**, **Upload from computer**, or **Google Drive** in the form and run the cell. For Drive, paste the image path into its field. Computer uploads transfer one image to the Colab runtime under a generated local name. Use images that you are permitted to process there.

| Preparation | Behavior |
|---|---|
| Require 480 x 480 | Default; reject other dimensions. |
| Resize square image | Accept a square image and resize it to 480 x 480. Reject rectangles. |
| Center crop, then resize | Crop a centered square with the shorter side, then resize to 480 x 480. Edge content may be removed. |
| White padding, then resize | Add white margins to a centered square with the longer side, then resize to 480 x 480. |
| Stretch to square (legacy) | Resize the whole image to 480 x 480; rectangular aspect ratios change. |

Resizing is bilinear. Cropping or padding an already square image only resizes it. With an odd difference, the extra cropped or padded pixel is on the right/bottom. Supported inputs are single-frame PNG, JPEG, BMP, TIFF or WebP; animated and multipage files are rejected. Oversized square padding is rejected before allocation.

Review the prepared-image preview. Source files are read without rewriting them. These preparation options do not establish that an arbitrary ECG image matches the paper's data preparation. After choosing another image, rerun the options and inference cells; a failed or cancelled selection cannot reuse the preceding result.

In [ ]:
# @title Choose and prepare one input image
INPUT_READY = False
OPTIONS_READY = False
IMAGE_PATH = None
PREPARED_IMAGE_PATH = None
input_record = None
INPUT_TOKEN = None
RESULT_INPUT_TOKEN = None
RESULT_OPTIONS_SIGNATURE = None
result = None
INPUT_SOURCE = "Synthetic example" # @param ["Synthetic example", "Upload from computer", "Google Drive"]
USER_IMAGE_PATH = "/content/drive/MyDrive/ecg_inference/data/example.png" # @param {type:"string"}
IMAGE_PREPARATION = "Require 480 x 480" # @param ["Require 480 x 480", "Resize square image", "Center crop, then resize", "White padding, then resize", "Stretch to square (legacy)"]

import base64
from html import escape
import tempfile
from uuid import uuid4
from IPython.display import HTML, display

check_execution_device()
POLICY_CHOICES = {
    "Require 480 x 480": "strict480", "Resize square image": "square-resize",
    "Center crop, then resize": "center-crop", "White padding, then resize": "pad",
    "Stretch to square (legacy)": "resize",
}
if IMAGE_PREPARATION not in POLICY_CHOICES:
    raise ValueError("Choose an image-preparation option from the menu.")
INPUT_POLICY = POLICY_CHOICES[IMAGE_PREPARATION]
INPUT_SELECTION = (INPUT_SOURCE, USER_IMAGE_PATH, IMAGE_PREPARATION, INPUT_POLICY)

def check_prepared_input():
    check_execution_device()
    if not globals().get("INPUT_READY", False) or not isinstance(globals().get("input_record"), dict):
        raise RuntimeError("Choose an image and complete input preparation successfully.")
    if INPUT_SELECTION != (INPUT_SOURCE, USER_IMAGE_PATH, IMAGE_PREPARATION, INPUT_POLICY):
        raise RuntimeError("The image selection changed. Rerun the input cell and inference options.")
    if IMAGE_PATH != PREPARED_IMAGE_PATH:
        raise RuntimeError("The input path changed. Choose the image again.")
    if IMAGE_PATH is None or not IMAGE_PATH.is_file() or any(p.is_symlink() for p in (IMAGE_PATH, *IMAGE_PATH.parents)):
        raise RuntimeError("The selected input is unavailable. Choose it again.")
    digest = hashlib.sha256()
    with IMAGE_PATH.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != input_record["input_sha256"]:
        raise RuntimeError("The input changed after preparation. Choose it again.")

input_folder = Path(tempfile.mkdtemp(prefix="ecg_input_", dir="/content"))
preview_path = input_folder / "prepared_preview.png"
helper_flags = []
if INPUT_SOURCE == "Synthetic example":
    pending_image = Path("/content/ecg_inference_synthetic.png")
    input_kind = "generated_synthetic_trace"
    helper_flags.append("--generate-synthetic")
elif INPUT_SOURCE == "Upload from computer":
    from google.colab import files
    if not callable(getattr(files, "upload_file", None)):
        raise RuntimeError("Single-file upload is unavailable in this Colab runtime. Start a current runtime or use Google Drive.")
    pending_image = input_folder / "uploaded_image.bin"
    print("Choose one image. Its bytes will be transferred to this Colab runtime.")
    try:
        files.upload_file(str(pending_image))
    except Exception:
        raise RuntimeError("Upload did not complete. Choose one image and rerun this cell.") from None
    input_kind = "user_upload"
elif INPUT_SOURCE == "Google Drive":
    pending_image = Path(USER_IMAGE_PATH).expanduser()
    if not pending_image.is_absolute():
        raise ValueError("Enter the full absolute Google Drive image path, starting with /content/drive/.")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    input_kind = "google_drive_file"
else:
    raise ValueError("Choose Synthetic example, Upload from computer or Google Drive.")

helper_command = [
    str(INFERENCE_PYTHON), "-c",
    "import sys; sys.path.insert(0, sys.argv.pop(1)); from takotsubo_ami_ai_ecg.colab_support import main; raise SystemExit(main())",
    str(REPO_DIR / "src"), "--image", str(pending_image), "--kind", input_kind,
    "--input-policy", INPUT_POLICY, "--preview", str(preview_path), *helper_flags,
]
prepared = subprocess.run(helper_command, capture_output=True, text=True)
if prepared.returncode:
    try:
        message = json.loads(prepared.stderr)["error"]
    except (ValueError, KeyError, TypeError):
        message = "Image preparation failed. Check the selected image, format and size policy."
    raise RuntimeError(message)
pending_record = json.loads(prepared.stdout)
if any(p.is_symlink() for p in (preview_path, *preview_path.parents)):
    raise RuntimeError("The input preview is not a regular file.")
preview_bytes = preview_path.read_bytes()
if (hashlib.sha256(preview_bytes).hexdigest() != pending_record["preview_sha256"]
        or len(preview_bytes) != pending_record["preview_size_bytes"]):
    raise RuntimeError("The prepared preview changed. Rerun this cell.")
IMAGE_PATH = pending_image
PREPARED_IMAGE_PATH = pending_image
input_record = pending_record
INPUT_TOKEN = uuid4().hex
INPUT_READY = True
check_prepared_input()
print("Input source:", INPUT_SOURCE)
print("Original dimensions:", input_record["size"], "| Format:", input_record["format"])
print("Preparation:", IMAGE_PREPARATION, "| Prepared dimensions:", input_record["prepared_size"])
if INPUT_POLICY == "center-crop":
    print("The centered square crop can remove content at the image edges. Review the prepared image below.")
elif INPUT_POLICY == "pad":
    print("The image is centered on a white square before resizing. Review the added margins below.")
elif INPUT_POLICY == "resize":
    print("Legacy stretch resizes the whole image and can change its aspect ratio.")
encoded_preview = base64.b64encode(preview_bytes).decode("ascii")
display(HTML('<figure style="margin:0;max-width:480px"><figcaption>Prepared model input (480 x 480)</figcaption>'
             f'<img alt="Prepared model input" src="data:image/png;base64,{encoded_preview}" style="width:100%;height:auto" /></figure>'))

## 5. Select inference options

**Paper defaults** uses ViT 24 views/magnitude 1, ConvNeXtV2 1 view and MLP-Mixer 32 views/magnitude 3. **No TTA** uses one unchanged view per model. **Custom** enables the six per-model fields below: views must be integers of at least 1; magnitude is an integer from 0 to 9 and is unused when views = 1. Views include the unchanged image. Reducing view counts reduces the number of model evaluations; reducing magnitude alone does not reduce that count.

CPU fallback keeps the selected TTA preset. The ensemble averages raw logits and then applies softmax. Defaults remain float32, seed 1234 and batch size 1; CPU uses zero workers, GPU uses four. Different TTA/worker/device choices are recorded and are not claimed to reproduce the historical evaluation exactly.

**ENABLE_GRADCAM = True** displays ViT and ConvNeXtV2 heatmaps and overlays in this notebook. Maps use separate unaugmented float32 passes, even when prediction uses TTA. There is no MLP-Mixer or combined ensemble map; disable Grad-CAM for MLP-Mixer alone. The default target is each model's TTA-predicted class. `ensemble-predicted` requires the ensemble model. The default normalization matches inference; `legacy-vit` is a comparison setting and does not establish historical-map reproduction.

After changing a form option, run the **Select the model, TTA and Grad-CAM** cell before running inference. Check the printed effective settings: **No TTA** must show one view for each model. Selecting a form value without executing its cell does not update the active Python settings.


In [ ]:
# @title Select the model, TTA and Grad-CAM
OPTIONS_READY = False
RESULT_OPTIONS_SIGNATURE = None
result = None
MODEL = "ensemble" # @param ["ensemble", "vit", "convnextv2", "mlpmixer"]
TTA_PRESET = "Paper defaults" # @param ["Paper defaults", "No TTA", "Custom"]
# The following six fields apply only to Custom.
CUSTOM_VIT_VIEWS = 24 # @param {type:"integer"}
CUSTOM_VIT_MAGNITUDE = 1 # @param {type:"integer"}
CUSTOM_CONVNEXTV2_VIEWS = 1 # @param {type:"integer"}
CUSTOM_CONVNEXTV2_MAGNITUDE = 5 # @param {type:"integer"}
CUSTOM_MLPMIXER_VIEWS = 32 # @param {type:"integer"}
CUSTOM_MLPMIXER_MAGNITUDE = 3 # @param {type:"integer"}
ENABLE_GRADCAM = True # @param {type:"boolean"}
CAM_TARGET = "model-predicted" # @param ["model-predicted", "ensemble-predicted", "tako", "AMI"]
CAM_NORMALIZATION = "inference" # @param ["inference", "legacy-vit"]
CAM_OPACITY = 0.6 # @param {type:"number"}
SEED = 1234 # @param {type:"integer"}

check_prepared_input()
from takotsubo_ami_ai_ecg.colab_support import resolve_tta
from takotsubo_ami_ai_ecg.settings import InferenceSettings, ModelSettings
TTA_SETTINGS = resolve_tta(
    TTA_PRESET, CUSTOM_VIT_VIEWS, CUSTOM_VIT_MAGNITUDE,
    CUSTOM_CONVNEXTV2_VIEWS, CUSTOM_CONVNEXTV2_MAGNITUDE,
    CUSTOM_MLPMIXER_VIEWS, CUSTOM_MLPMIXER_MAGNITUDE,
)
VIT_TTA, VIT_MAGNITUDE = TTA_SETTINGS["vit"]["tta"], TTA_SETTINGS["vit"]["magnitude"]
CONVNEXTV2_TTA, CONVNEXTV2_MAGNITUDE = TTA_SETTINGS["convnextv2"]["tta"], TTA_SETTINGS["convnextv2"]["magnitude"]
MLPMIXER_TTA, MLPMIXER_MAGNITUDE = TTA_SETTINGS["mlpmixer"]["tta"], TTA_SETTINGS["mlpmixer"]["magnitude"]
OUTPUT_DIR = Path("/content/ecg_inference_results")
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0]
THRESHOLD = None
PRECISION = "float32"
BATCH_SIZE = 1
NUM_WORKERS = 0 if DEVICE == "cpu" else 4

if MODEL not in ("ensemble", "vit", "convnextv2", "mlpmixer"):
    raise ValueError("Select a supported prediction model.")
if ENABLE_GRADCAM and MODEL == "mlpmixer":
    raise ValueError("Grad-CAM is available for ViT and ConvNeXtV2. Disable it when selecting MLP-Mixer alone.")
if CAM_TARGET == "ensemble-predicted" and MODEL != "ensemble" and ENABLE_GRADCAM:
    raise ValueError("The ensemble-predicted Grad-CAM target requires the ensemble model.")
if not isinstance(CAM_OPACITY, (int, float)) or isinstance(CAM_OPACITY, bool) or not 0 <= CAM_OPACITY <= 1:
    raise ValueError("Grad-CAM opacity must be between 0 and 1.")
InferenceSettings(
    vit=ModelSettings(VIT_TTA, VIT_MAGNITUDE), convnextv2=ModelSettings(CONVNEXTV2_TTA, CONVNEXTV2_MAGNITUDE),
    mlpmixer=ModelSettings(MLPMIXER_TTA, MLPMIXER_MAGNITUDE), ensemble_weights=tuple(ENSEMBLE_WEIGHTS),
    threshold=THRESHOLD, seed=SEED, input_policy=INPUT_POLICY, precision=PRECISION,
    device=DEVICE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
).validate()

def current_options_signature():
    return json.dumps({
        "model": MODEL, "device": DEVICE, "precision": PRECISION, "input_policy": INPUT_POLICY,
        "tta_preset": TTA_PRESET,
        "custom_fields": [CUSTOM_VIT_VIEWS, CUSTOM_VIT_MAGNITUDE, CUSTOM_CONVNEXTV2_VIEWS,
                          CUSTOM_CONVNEXTV2_MAGNITUDE, CUSTOM_MLPMIXER_VIEWS, CUSTOM_MLPMIXER_MAGNITUDE],
        "tta": [VIT_TTA, VIT_MAGNITUDE, CONVNEXTV2_TTA, CONVNEXTV2_MAGNITUDE, MLPMIXER_TTA, MLPMIXER_MAGNITUDE],
        "ensemble_weights": ENSEMBLE_WEIGHTS, "threshold": THRESHOLD, "seed": SEED,
        "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
        "gradcam": [ENABLE_GRADCAM, CAM_TARGET, CAM_NORMALIZATION, CAM_OPACITY],
    }, sort_keys=True, allow_nan=False)

OPTIONS_SIGNATURE = current_options_signature()
OPTIONS_READY = True

def check_prepared_options():
    check_prepared_input()
    if not globals().get("OPTIONS_READY", False) or current_options_signature() != OPTIONS_SIGNATURE:
        raise RuntimeError("Inference options changed or are incomplete. Rerun the options cell.")

def current_weight_paths():
    if not globals().get("WEIGHTS_READY", False):
        raise RuntimeError("Complete weight preparation before inference.")
    names = ("vit", "convnextv2", "mlpmixer") if MODEL == "ensemble" else (MODEL,)
    paths = [WEIGHTS_DIR / name / "model.safetensors" for name in names]
    if "vit" in names:
        paths.append(WEIGHTS_DIR / "vit" / "config.json")
    if any(not path.is_file() for path in paths):
        raise RuntimeError("A selected model weight or configuration file is missing.")
    return paths

def check_current_result():
    check_prepared_options()
    if (not isinstance(globals().get("result"), dict)
            or globals().get("RESULT_INPUT_TOKEN") != INPUT_TOKEN
            or globals().get("RESULT_OPTIONS_SIGNATURE") != OPTIONS_SIGNATURE):
        raise RuntimeError("Run inference successfully for the current image and options before viewing or downloading results.")

print("Actual device:", DEVICE, "| Precision:", PRECISION, "| Workers:", NUM_WORKERS)
print("TTA preset:", TTA_PRESET)
print(json.dumps(TTA_SETTINGS, indent=2))
if DEVICE == "cpu":
    print("WARNING: CPU inference can be slow. Select No TTA or reduce Custom view counts if needed; your settings are not changed automatically.")
print("Grad-CAM:", "enabled" if ENABLE_GRADCAM else "disabled")

## 6. Check the input and run inference

The next cells verify the selected input and settings, then run the shared CLI implementation. The raw input is passed to the CLI with the selected preparation policy; the preview is not resized a second time. Cropping, padding and resizing are also available through CLI `--input-policy`.

Changing the image or rerunning the options cell invalidates the current result. Run inference again before viewing or downloading it. Each run creates new output filenames, and prior saved files remain available.

Progress shows the current model, image/view count and elapsed time; it is not a time estimate. CPU runs can take substantially longer. Logs and errors appear as they arrive. Stopping the cell stops its inference subprocess. Completion is reported after both the prediction and matching run record have been saved.

In [ ]:
check_prepared_options()
required_weights = current_weight_paths()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Prepared input dimensions:", input_record["prepared_size"])
print("Preparation:", INPUT_POLICY)
print("Selected model:", MODEL, "| Device:", DEVICE)
print("Input and selected-weight checks passed.")

In [ ]:
result = None
RESULT_INPUT_TOKEN = None
RESULT_OPTIONS_SIGNATURE = None

from datetime import datetime, timezone
import hashlib
from html import escape
import math
import time
import uuid
from IPython.display import HTML, display

check_prepared_options()
required_weights = current_weight_paths()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_JSON = OUTPUT_DIR / f"prediction_{MODEL}_{stamp}_{uuid.uuid4().hex[:8]}.json"
RUN_JSON = OUTPUT_JSON.with_suffix(".colab_run.json")

command = [
    str(INFERENCE_PYTHON), "-u",
    str(REPO_DIR / "examples" / f"predict_{MODEL}.py"),
    "--image", str(IMAGE_PATH),
    "--weights-dir", str(WEIGHTS_DIR),
    "--output", str(OUTPUT_JSON),
    "--vit-tta", str(VIT_TTA), "--vit-magnitude", str(VIT_MAGNITUDE),
    "--convnextv2-tta", str(CONVNEXTV2_TTA), "--convnextv2-magnitude", str(CONVNEXTV2_MAGNITUDE),
    "--mlpmixer-tta", str(MLPMIXER_TTA), "--mlpmixer-magnitude", str(MLPMIXER_MAGNITUDE),
    "--ensemble-weights", *[str(value) for value in ENSEMBLE_WEIGHTS],
    "--seed", str(SEED),
    "--device", DEVICE,
    "--precision", PRECISION,
    "--input-policy", INPUT_POLICY,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--progress", "json",
]
if ENABLE_GRADCAM:
    command.extend(["--gradcam", "--cam-target", CAM_TARGET,
                    "--cam-normalization", CAM_NORMALIZATION, "--cam-opacity", str(CAM_OPACITY)])
if THRESHOLD is not None:
    command.extend(["--threshold", str(THRESHOLD)])

progress_prefix = "ECG_PROGRESS "
progress_state = {"completed": 0, "total": 0, "message": "Starting inference."}
progress_display = None
process = None
progress_error = False
started = time.monotonic()


def finite_number(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default


def count_pair(current, total):
    if current is None or total is None:
        return "n/a"
    return f"{escape(str(current))}/{escape(str(total))}"


def render_progress(message=None):
    global progress_display
    total = max(0.0, finite_number(progress_state.get("total")))
    completed = max(0.0, finite_number(progress_state.get("completed")))
    percent = min(100.0, 100.0 * completed / total) if total else 0.0
    elapsed = max(time.monotonic() - started, finite_number(progress_state.get("elapsed_seconds")))
    model = progress_state.get("model") or "n/a"
    model_position = count_pair(progress_state.get("model_index"), progress_state.get("model_count"))
    tta = count_pair(progress_state.get("tta_index"), progress_state.get("tta_total"))
    images = count_pair(progress_state.get("images_completed"), progress_state.get("images_total"))
    status = escape(str(message if message is not None else progress_state.get("message", "Running inference.")))
    content = HTML(
        '<div style="padding:10px 0;max-width:760px">'
        f'<div><strong>{status}</strong></div>'
        f'<progress value="{percent:.2f}" max="100" style="width:100%;height:20px" '
        'aria-label="Inference progress"></progress>'
        f'<div>{percent:.1f}% | Model: {escape(str(model))} ({model_position}) '
        f'| TTA: {tta} | Images: {images} | Elapsed: {elapsed:.1f} s</div></div>'
    )
    if progress_display is None:
        progress_display = display(content, display_id=True)
    else:
        progress_display.update(content)


def gradcam_artifact_paths(prediction_result):
    """Accept only documented files in this run's output directory."""
    artifacts = []
    for row in prediction_result["predictions"]:
        cam = row.get("gradcam")
        if cam is None:
            continue
        sample_id = row.get("sample_id", "")
        if not isinstance(sample_id, str) or not re.fullmatch(r"sample_[0-9]{6}", sample_id):
            raise RuntimeError("Unexpected visualization sample identifier.")
        stem = f"{OUTPUT_JSON.stem}_{sample_id}"
        if cam.get("input_image") != f"{stem}_input.png":
            raise RuntimeError("Unexpected visualization input filename.")
        names = [cam["input_image"]]
        selected = set(row["models"]) & {"vit", "convnextv2"}
        if set(cam.get("models", {})) != selected:
            raise RuntimeError("The visualization model set does not match the prediction.")
        for model in sorted(selected):
            expected = {"heatmap": f"{stem}_{model}_heatmap.png",
                        "overlay": f"{stem}_{model}_overlay.png",
                        "values": f"{stem}_{model}_cam.npy"}
            if cam["models"][model].get("files") != expected:
                raise RuntimeError("Unexpected visualization artifact filenames.")
            names.extend(expected.values())
        for name in names:
            path = OUTPUT_JSON.parent / name
            if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
                raise RuntimeError("A visualization artifact is missing or is a symbolic link.")
            if path.stat().st_size > 10 * 1024 * 1024:
                raise RuntimeError("A visualization artifact exceeds the expected size.")
            artifacts.append(path)
    if len({path.name for path in artifacts}) != len(artifacts):
        raise RuntimeError("Visualization artifact names are not unique.")
    return artifacts


def stop_inference_process():
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)


try:
    render_progress()
    process = subprocess.Popen(
        command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        if line.startswith(progress_prefix):
            try:
                event = json.loads(line[len(progress_prefix):])
            except json.JSONDecodeError:
                print("An unreadable progress update was skipped.", flush=True)
                continue
            if not isinstance(event, dict) or event.get("event") not in {"stage", "progress", "complete", "error"}:
                print("An unrecognized progress update was skipped.", flush=True)
                continue
            progress_state.update(event)
            if event["event"] == "error":
                progress_error = True
                render_progress("Inference failed. Review the logs below.")
            elif event["event"] == "complete":
                render_progress("Prediction saved; finishing the run record.")
            else:
                render_progress()
        else:
            print(line, end="", flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(
            f"Inference stopped with exit code {returncode}. Review the error displayed above."
        )
    if progress_error:
        raise RuntimeError("Inference reported an error. Review the logs above.")

    pending_result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))

    if pending_result["metadata"]["actual_device"] != DEVICE:
        raise RuntimeError("Inference did not use the resolved execution device.")
    if any(pending_result["metadata"]["versions"][name] != version
           for name, version in EXPECTED_VERSIONS.items() if name not in {"setuptools", "pip"}):
        raise RuntimeError("Inference did not use the reviewed inference versions.")
    if len(pending_result["predictions"]) != 1 or pending_result["predictions"][0]["input_sha256"] != input_record["input_sha256"]:
        raise RuntimeError("Inference did not use the recorded input image.")
    if hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest() != input_record["input_sha256"]:
        raise RuntimeError("The input changed during inference.")
    if ENABLE_GRADCAM and any("gradcam" not in row for row in pending_result["predictions"]):
        raise RuntimeError("Grad-CAM was requested but its artifacts were not recorded.")
    artifact_paths = gradcam_artifact_paths(pending_result)
    artifact_records = [{"filename": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                         "size_bytes": path.stat().st_size} for path in artifact_paths]
    if artifact_paths:
        cli_artifacts = pending_result["metadata"]["gradcam"].get("artifacts", [])
        if sorted(cli_artifacts, key=lambda item: item["filename"]) != sorted(artifact_records, key=lambda item: item["filename"]):
            raise RuntimeError("The visualization files do not match the CLI's saved artifact hashes.")
    # Link the output to its run record without storing an absolute input path.
    run_record = {
        "code_commit": CODE_COMMIT,
        "source_ref": GIT_REF,
        "source_sha256": SOURCE_SHA256,
        "input": input_record,
        "prediction_file": OUTPUT_JSON.name,
        "prediction_sha256": hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest(),
        "runtime": runtime_versions,
        "inference_versions": pending_result["metadata"]["versions"],
        "actual_device": pending_result["metadata"]["actual_device"],
        "requested_device": REQUESTED_DEVICE,
        "tta_preset": TTA_PRESET,
        "gradcam": pending_result["metadata"].get("gradcam"),
        "artifacts": artifact_records,
    }
    from takotsubo_ami_ai_ecg.safe_output import write_files
    run_bytes = (json.dumps(run_record, ensure_ascii=False, indent=2, allow_nan=False) + "\n").encode("utf-8")
    write_files([(RUN_JSON, run_bytes)], protected_paths=[IMAGE_PATH, OUTPUT_JSON, *required_weights, *artifact_paths])
    check_prepared_options()
    result = pending_result
    RESULT_INPUT_TOKEN = INPUT_TOKEN
    RESULT_OPTIONS_SIGNATURE = OPTIONS_SIGNATURE
    render_progress("Inference completed; prediction and run record saved.")
except BaseException as exc:
    result = None
    try:
        stop_inference_process()
    finally:
        failure_message = "Inference cancelled." if isinstance(exc, KeyboardInterrupt) else "Inference failed. Review the error below."
        try:
            render_progress(failure_message)
        except Exception:
            print(failure_message, flush=True)
    raise
finally:
    if process is not None and process.stdout is not None:
        process.stdout.close()

print("Inference completed.")
print("Prediction:", OUTPUT_JSON.name)
print("Run record:", RUN_JSON.name)

The next cells show the saved prediction and optional Grad-CAM overlays. The final download cell creates **one ZIP containing the prediction JSON, matching run-record JSON and all visualization artifacts from that run**. With Grad-CAM disabled, it contains only the two JSON files. With Grad-CAM enabled, it also contains a prepared input-image copy; keep this ZIP private when the input is private. Model weights are never included.


## 7. Review the results

The class order is **0 = tako, 1 = AMI**. By default, the ensemble uses `prob_AMI >= 0.5`, while individual models use argmax (a probability tie is assigned to AMI for the ensemble and tako for an individual model). If you set a numeric `THRESHOLD`, it is used for either case. The following cell displays model outputs. Without comparison against ground-truth labels, these outputs do not establish sensitivity, specificity, or accuracy.

The JSON records the actual execution settings, weight identifiers, logits, and probabilities. Colab's working storage is temporary, so save any results you need to retain.
Successful inference creates `prediction_<model>_<timestamp>_<id>.json` and a matching file ending in `.colab_run.json`. The latter records the code commit, runtime environment, and SHA-256 of the prediction file. The run record also links visualization files by their names and SHA-256. Use the download cell below to save the complete run.


In [ ]:
check_current_result()

if not isinstance(globals().get("result"), dict):
    raise RuntimeError("Complete inference successfully before reviewing the results.")

for prediction in result["predictions"]:
    summary = {
        "sample_id": prediction.get("sample_id"),
        "pred_id": prediction["pred_id"],
        "pred_label": prediction["pred_label"],
        "logits": prediction["logits"],
        "probabilities": prediction["probabilities"],
        "models": prediction.get("models", {}),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))

## 8. Review Grad-CAM

These maps show **relative positive contributions to one class logit on the unaugmented image**. Prediction probabilities above may include TTA; the heatmap is not an explanation of the entire TTA aggregation. Its color value is not a class probability or a calibrated importance score, and maps from different models are scaled independently.

The blue-to-yellow palette runs from lower to higher map values. A constant map is shown in neutral gray and its overlay leaves the image unchanged. Each panel identifies the selected class and normalization. The optional `legacy-vit` setting is for comparison with the old ViT preprocessing; it is not a claim that the historical reader-study images have been reproduced.


In [ ]:
check_current_result()

if not isinstance(globals().get("result"), dict):
    raise RuntimeError("Complete inference successfully before viewing Grad-CAM.")

import base64
from html import escape
from IPython.display import HTML, display

saved_record = json.loads(RUN_JSON.read_text(encoding="utf-8"))
if saved_record.get("prediction_sha256") != hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest():
    raise RuntimeError("The saved prediction changed after the run record was created.")
saved_result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
paths = gradcam_artifact_paths(saved_result)
expected_artifacts = {item["filename"]: item for item in saved_record.get("artifacts", [])}
for path in paths:
    item = expected_artifacts.get(path.name, {})
    data = path.read_bytes()
    if item.get("sha256") != hashlib.sha256(data).hexdigest() or item.get("size_bytes") != len(data):
        raise RuntimeError("A visualization file does not match its saved run record.")

def png_panel(filename, title):
    data = (OUTPUT_JSON.parent / filename).read_bytes()
    encoded = base64.b64encode(data).decode("ascii")
    return (f'<figure style="margin:0;max-width:480px"><figcaption>{escape(title)}</figcaption>'
            f'<img alt="{escape(title)}" src="data:image/png;base64,{encoded}" '
            'style="width:100%;height:auto" /></figure>')

for prediction in saved_result["predictions"]:
    cam = prediction.get("gradcam")
    if cam is None:
        print("Grad-CAM was disabled for this run.")
        continue
    display(HTML(png_panel(cam["input_image"], "Prepared input (480 x 480)")))
    for model, details in cam["models"].items():
        heading = f"{model}: target {details['target_label']} ({details['target_source']})"
        print(heading)
        print("Unaugmented logits [tako, AMI]:", details["unaugmented_logits"])
        print("Normalization:", details["normalization"])
        if details.get("map_is_constant"):
            print("Constant map: no spatial ranking is available; the overlay is unchanged.")
        panels = png_panel(details["files"]["heatmap"], "Relative Grad-CAM: low (blue) to high (yellow)")
        panels += png_panel(details["files"]["overlay"], f"{model} overlay; target {details['target_label']}")
        display(HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + panels + '</div>'))

In [ ]:
# Download this run's linked JSON records and optional visualizations as one ZIP.
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    check_current_result()
    import hashlib
    import json
    from io import BytesIO
    from uuid import uuid4
    from zipfile import ZIP_DEFLATED, ZipFile
    from google.colab import files
    from takotsubo_ami_ai_ecg.safe_output import write_files

    if not isinstance(globals().get("result"), dict) or not OUTPUT_JSON.is_file() or not RUN_JSON.is_file():
        raise RuntimeError("Confirm that inference completed and both JSON files were saved.")
    if any(p.is_symlink() for path in (OUTPUT_JSON, RUN_JSON) for p in (path, *path.parents)):
        raise RuntimeError("Result files must not use symbolic links.")
    prediction_bytes = OUTPUT_JSON.read_bytes()
    run_bytes = RUN_JSON.read_bytes()
    record = json.loads(run_bytes)
    if (record.get("prediction_file") != OUTPUT_JSON.name
            or record.get("prediction_sha256") != hashlib.sha256(prediction_bytes).hexdigest()):
        raise RuntimeError("The prediction and run record do not match. Check the selected files.")
    saved_prediction = json.loads(prediction_bytes)
    artifact_paths = gradcam_artifact_paths(saved_prediction)
    artifacts = record.get("artifacts", [])
    if (not isinstance(artifacts, list) or len(artifacts) != len(artifact_paths)
            or [item.get("filename") for item in artifacts] != [path.name for path in artifact_paths]):
        raise RuntimeError("The visualization files do not match the run record.")
    contents = [(OUTPUT_JSON.name, prediction_bytes), (RUN_JSON.name, run_bytes)]
    for path, item in zip(artifact_paths, artifacts):
        data = path.read_bytes()
        if (hashlib.sha256(data).hexdigest() != item.get("sha256")
                or len(data) != item.get("size_bytes")):
            raise RuntimeError("A visualization file changed after the run record was saved.")
        contents.append((path.name, data))

    with BytesIO() as buffer:
        with ZipFile(buffer, "w", compression=ZIP_DEFLATED) as archive:
            for name, data in contents:
                archive.writestr(name, data)
        zip_bytes = buffer.getvalue()

    ZIP_PATH = OUTPUT_JSON.parent / f"{OUTPUT_JSON.stem}_{uuid4().hex[:8]}.zip"
    write_files([(ZIP_PATH, zip_bytes)], protected_paths=[OUTPUT_JSON, RUN_JSON, *artifact_paths])
    print("ZIP prepared:", ZIP_PATH.name)
    print("Included:", ", ".join(name for name, _ in contents))
    files.download(str(ZIP_PATH))

## Validation scope

The input-selection, geometry, CPU fallback and TTA controls above are included in v0.4.0. Local Windows CPU validation completed on 2026-10-29 (JST; 2026-10-28 UTC): 169 of 172 tests passed, three unavailable-hardlink cases were skipped, and there were no failures or errors. The existing environment used Python 3.13.15, torch 2.13.0+cpu, torchvision 0.28.0+cpu, transformers 5.10.4 and an isolated filelock 4.2.1 overlay.

On one generated 480 x 480 image with paper TTA and the selected weights, all three models' and the ensemble's logits and probabilities matched a frozen saved run in the same environment exactly. Both ViT/ConvNeXtV2 numerical Grad-CAM maps and the prepared input pixels also matched exactly. Each new geometry policy (`square-resize`, `center-crop`, `pad`) was separately compared with an independently prepared Pillow image using selected-weight ViT without TTA; prepared pixels, logits, probabilities and numerical Grad-CAM map matched exactly. The original environment, inputs/source and weights were preserved.

On 2026-10-29 (JST), eight runs of candidate `0.4.0.dev0` completed in Colab. A100/Python 3.13.16/CUDA float32 runs covered the generated strict480 input, a 720 x 720 computer upload with square-resize, a 721 x 480 upload with center-crop and pad in separate runs, and a 480 x 480 Google Drive image, all with paper TTA, batch 1 and four workers. CPU/Python 3.13.18/float32 runs automatically selected CPU with CUDA unavailable and used the generated strict480 image twice with paper TTA and once with No TTA (1/1/1), batch 1 and zero workers. Each run enabled Grad-CAM and saved the matching nine-file ZIP.

The candidate source archive and all 118 recorded source-file hashes matched. Recorded official weight/configuration identities, prediction arithmetic, input/preparation correspondence, finite float32 480 x 480 maps, and saved-file hashes/sizes were checked. The Drive input differed from the CPU generated image, so these runs are not a controlled timing comparison. The public notebook uses Git checkout in place of candidate archive loading; all subsequent code cells are unchanged. The public-tag source acquisition and execution check completed on 2026-10-29 as described below. These checks do not establish clinical performance, exhaustive custom-setting coverage, historical stochastic-TTA reproduction or reader-study visualization equivalence. Earlier checks below apply to their recorded versions.

**Public v0.4.0 follow-up, 2026-10-29 (JST).** The published notebook completed source acquisition, inference, Grad-CAM and result export on one generated 480 x 480 RGB PNG using A100, Python 3.13.16, CUDA float32, strict480, paper TTA (ViT 24/magnitude 1; ConvNeXtV2 1 unchanged view; MLP-Mixer 32/magnitude 3), seed 1234, batch 1 and four workers. The run recorded `source_ref = v0.4.0`, package 0.4.0 and commit `6a16326b761c198c11776155102dc31d1308bfeb`. Core inference versions were torch 2.13.0+cu130, torchvision 0.28.0+cu130, timm 1.0.22, transformers 5.10.4, numpy 2.2.6, Pillow 13.0.0, safetensors 0.7.0 and filelock 4.2.1.

All 21 recorded inference-source hashes, three official checkpoint hashes and the ViT configuration hash matched. The matching prediction/run JSON pair, five PNGs and two NPY arrays passed integrity and correspondence checks. Independent recalculation confirmed the raw-logit ensemble, softmax, prepared input pixels and heatmap/overlay rendering. Both numerical maps were finite float32 480 x 480 arrays.

Compared with the saved candidate A100 run using the same generated input and inference settings, all model and ensemble logits and probabilities matched exactly. The ConvNeXtV2 map matched exactly. The ViT map had a maximum absolute difference of `2.384185791015625e-7`; both models' heatmap and overlay pixels were identical to the candidate. This public-tag run did not include a paired Grad-CAM-off comparison or repeat the candidate CPU/upload/Drive/geometry checks. It completes the public-tag workflow check for this synthetic example, not clinical or historical reader-study validation.

Post-release verification is maintained in the [current implementation status](https://github.com/ekagawa007/takotsubo_ami_ai_ecg/blob/main/docs/implementation_status.md#v040-image-input-and-colab-controls). Existing release tags and their notebook snapshots retain the documentation available at publication.

The v0.2.1 prediction baseline passed a WSL/RTX 5090 float32 no-TTA comparison on ten private ECG images, with no changed classifications and all outputs within the recorded tolerance. Its final public-tag Colab/A100 runs also completed the synthetic-image example with paper TTA; source, selected-weight identities and saved result records were verified. Those baseline checks are separate from the Grad-CAM checks below.

On 2026-10-23, Grad-CAM candidate `0.3.0.dev0` completed 119 local Windows CPU tests: 116 passed and three hard-link tests were skipped because the selected temporary directory did not support them. Using the three selected checkpoints and one generated 480 x 480 image, paired CPU float32 checks passed without TTA and with paper TTA (ViT 24/magnitude 1, ConvNeXtV2 1, MLP-Mixer 32/magnitude 3). A paired WSL/RTX 5090 CUDA float32 check also passed with paper TTA, Python 3.12.12, batch size 1 and zero workers. In each paired check, enabling Grad-CAM left all prediction values exactly unchanged.

The candidate also completed Colab/A100 CUDA float32 paper-TTA inference and Grad-CAM on one generated image, using Python 3.13.15, batch size 1 and four workers. Both GPU workflows produced two finite float32 480 x 480 maps with verified PNG/NPY artifacts. The Colab candidate archive and all 111 recorded source hashes matched; its downloaded ZIP contained the matching two JSON records and seven visualization artifacts. This Colab run did not include a paired Grad-CAM-off comparison.

The WSL and Colab GPU checks used torch 2.13.0+cu130, torchvision 0.28.0+cu130, transformers 5.10.4 and Pillow 13.0.0. Release `0.3.0` differed from the tested candidate only in documentation and package-version metadata.

On 2026-10-24, the published `v0.3.0` workflow completed paper-TTA inference and Grad-CAM on Colab/A100 using Python 3.13.16, the same core inference versions, CUDA float32, one generated 480 x 480 image, batch size 1 and four workers. The run recorded commit `341ac955353f535fb74a4b6d8005a8b3b1290ad4`; all 20 recorded critical source hashes, the three official checkpoint hashes and the ViT configuration hash matched. Independent recalculation confirmed each model's softmax and the equal-weight raw-logit ensemble. Both maps were finite float32 480 x 480 arrays, and all nine saved files (two JSONs, five PNGs and two NPYs) passed integrity and correspondence checks. This public Colab run did not include a paired Grad-CAM-off comparison. See [implementation status](https://github.com/ekagawa007/takotsubo_ami_ai_ecg/blob/main/docs/implementation_status.md) for the complete validation record.

On 2026-10-28, the `v0.3.1` candidate completed WSL/RTX 5090 checks with Python 3.12.12 and filelock 4.2.1: 117 of 119 tests passed and two Windows-only cases were skipped. With one generated image, paper TTA, CUDA float32, batch 1 and zero workers, the filelock substitution preserved all prediction values and both numerical Grad-CAM maps exactly. The original environment was retained.

The matching candidate archive also completed this workflow in a fresh Colab/A100 inference environment with Python 3.13.16, CUDA float32, filelock 4.2.1, paper TTA, batch 1 and four workers. Its 112 source hashes, selected weights/configuration, synthetic RGB pixels, prediction calculation and all nine output files were verified. Both finite float32 480 x 480 maps matched their saved heatmaps and overlays. This Colab run did not include a paired Grad-CAM-off comparison. The public notebook uses Git checkout in place of candidate archive loading; all subsequent code cells are unchanged. The separate public-tag source acquisition and execution check completed on 2026-10-28 as described below. These candidate results do not establish historical paper-TTA or reader-study visualization equivalence.

On 2026-10-28, the published `v0.3.1` notebook completed source acquisition, paper-TTA prediction and Grad-CAM on Colab/A100 with Python 3.13.16, CUDA float32, filelock 4.2.1, one generated 480 x 480 RGB image, batch 1, four workers and seed 1234. The saved run recorded `v0.3.1` and commit `aa1c8f4596b63bed39c53e390e67318d1836b7f7`; all 20 recorded inference-source hashes, three official checkpoint hashes and the ViT configuration hash matched. The two JSONs and seven visualization artifacts passed integrity and correspondence checks; prediction arithmetic and map rendering were independently recalculated. All model and ensemble logits, probabilities and classes matched the candidate exactly. The ConvNeXtV2 map was byte-identical; the ViT map's maximum absolute difference was `2.384185791015625e-7`, with one 8-bit-level difference in one RGB component of each rendered image. Both maps were finite, nonconstant float32 480 x 480 arrays. The public-tag workflow check is complete for this synthetic image; no paired Grad-CAM-off comparison was performed in this Colab run.

Grad-CAM uses separate unaugmented float32 passes and does not explain the complete TTA average or three-model ensemble. The layer and gradient calculations follow the archived two-model visualization code, with current-inference normalization by default. ViT legacy preprocessing, display palette and raster export differ as documented in `docs/gradcam.md`. Historical reader-study heatmap equivalence remains unverified.

These software checks do not establish clinical performance or numerical equivalence to historical stochastic paper-TTA runs. TTA results depend on image order, batch size, worker count, library versions and precision.

The run record identifies the source, critical source files, input hashes, actual inference environment and exported visualization hashes. It does not contain the absolute input path. Keep records and image-containing result ZIPs private when their associated images are private, and clear notebook outputs before sharing an executed notebook.